In [1]:
import pyarrow.feather as feather

table = feather.read_table('dataset/data_andre.feather', memory_map=True)
df= table.to_pandas()
df.head()

,date,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,promo_value_DISC,promo_type_CIRC,promo_value_CIRC,promo_type_CIRE,promo_value_CIRE,promo_type_CLCP,promo_value_CLCP,promo_type_LFPE,promo_value_LFPE,store_id
0,2021-01-23,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0.0000,0,0.0,0,0.0,0,0.0,0,0.0,6269
1,2021-01-23,952568,6,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0.0000,0,0.0,0,0.0,0,0.0,1,0.0,6269
2,2021-01-23,809,13,juices drnks shelf stbl,pos subd grocery other,pos dept grocery,juice/aseptic/new age,0,0.0,1,...,0.0000,0,0.0,0,0.0,0,0.0,0,0.0,6269
3,2021-01-23,20405,3,dairy chs,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0.1228,0,0.0,0,0.0,0,0.0,0,0.0,6269
4,2021-01-23,605573,5,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0.0000,0,0.0,0,0.0,0,0.0,0,0.0,6269


In [2]:
df.duplicated().sum()

0

In [3]:
import pandas as pd

# 0) Clean dates
df["date"] = pd.to_datetime(df["date"]).dt.normalize()

# 1) Collapse to daily store totals (handles multiple items per day)
daily = (
    df.groupby(["store_id", "date"], as_index=False)["value"]
      .sum()
)

# --- Option A: working day = any activity row (nonzero sales) ---
open_days = daily[daily["value"] > 0]

# Per-store list of working dates
working_days_by_store = (
    open_days.groupby("store_id")["date"]
             .apply(lambda s: sorted(s.unique()))
             .to_dict()
)

# Example: working days for a specific store_id
store_id = 6269
working_days_store = working_days_by_store.get(store_id, [])

print(f"{store_id} has {len(working_days_store)} working days from {open_days['date'].min().date()} to {open_days['date'].max().date()}")
print(working_days_store[:365])  # peek first 10 dates

6269 has 760 working days from 2021-01-23 to 2023-02-22
[Timestamp('2021-01-23 00:00:00'), Timestamp('2021-01-24 00:00:00'), Timestamp('2021-01-25 00:00:00'), Timestamp('2021-01-26 00:00:00'), Timestamp('2021-01-27 00:00:00'), Timestamp('2021-01-28 00:00:00'), Timestamp('2021-01-29 00:00:00'), Timestamp('2021-01-30 00:00:00'), Timestamp('2021-01-31 00:00:00'), Timestamp('2021-02-01 00:00:00'), Timestamp('2021-02-02 00:00:00'), Timestamp('2021-02-03 00:00:00'), Timestamp('2021-02-04 00:00:00'), Timestamp('2021-02-05 00:00:00'), Timestamp('2021-02-06 00:00:00'), Timestamp('2021-02-07 00:00:00'), Timestamp('2021-02-08 00:00:00'), Timestamp('2021-02-09 00:00:00'), Timestamp('2021-02-10 00:00:00'), Timestamp('2021-02-11 00:00:00'), Timestamp('2021-02-12 00:00:00'), Timestamp('2021-02-13 00:00:00'), Timestamp('2021-02-14 00:00:00'), Timestamp('2021-02-15 00:00:00'), Timestamp('2021-02-16 00:00:00'), Timestamp('2021-02-17 00:00:00'), Timestamp('2021-02-18 00:00:00'), Timestamp('2021-02-19 00:

In [4]:
is_christmas = (daily["date"].dt.month == 12) & (daily["date"].dt.day == 25)
daily.loc[is_christmas & (daily["store_id"] == store_id)]

,store_id,date,value
336,6269,2021-12-25,2
701,6269,2022-12-25,0


In [5]:
import holidays
import pandas as pd

# make sure date column is datetime
df["date"] = pd.to_datetime(df["date"]).dt.normalize()

# define the full date range of your dataset
min_date, max_date = df["date"].min(), df["date"].max()

# generate US federal holidays within that range
us_holidays = holidays.UnitedStates(years=range(min_date.year, max_date.year + 1), observed=True)
holiday_dates = [pd.Timestamp(d) for d in us_holidays.keys() if min_date <= pd.Timestamp(d) <= max_date]

daily = (
    df.groupby(["store_id", "date"], as_index=False)["value"]
      .sum()
      .rename(columns={"value": "sales"})
)

# .loc only on holidays
daily_holidays = daily.loc[daily["date"].isin(holiday_dates)]
daily_holidays

,store_id,date,sales
23,6269,2021-02-15,8877
128,6269,2021-05-31,8573
146,6269,2021-06-18,9654
147,6269,2021-06-19,10513
162,6269,2021-07-04,8984
163,6269,2021-07-05,9581
226,6269,2021-09-06,9898
261,6269,2021-10-11,9386
292,6269,2021-11-11,8776
306,6269,2021-11-25,3393


In [6]:
df["store_id"].value_counts()

store_id
6269    1082371
Name: count, dtype: int64

In [7]:
len(df["item_id"].unique())

1427

In [8]:
df["item_id"].value_counts()

item_id
27        761
901498    761
6032      761
901555    761
6035      761
         ... 
79879     595
151824    591
210048    581
151827    578
151825    569
Name: count, Length: 1427, dtype: int64

In [9]:
df["cat_label"].unique()

array(['refrigerated drnks', 'carbonated sft drnks',
       'juices drnks shelf stbl', 'dairy chs', 'sprd btr mrgrn', 'yogurt',
       'rte cereal', 'sour cream', 'milknplant based bevs',
       'bottled water', 'dy non dy crm', 'eggs egg substitutes',
       'sparkling seltzer mixer', 'dairy cream', 'refrig dsrts',
       'refrigerated baked gds', 'new age bevs', 'hot cereal', 'itln chs',
       'sprt drnk', 'cottage chs', 'cream chs', 'dips refrigerated',
       'aseptic'], dtype=object)